In [6]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import os

try:
    from google.colab import auth
    import gspread
    from google.auth import default
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
os.chdir("/content/drive/MyDrive/Itai Stuff/CentralAsiaSnowpack/ISSW/Field_methods_to_track_instability_trends")

In [9]:
# --- CONFIGURATION ---

STUDY_LAT = 39.80515
STUDY_LON = -105.77977

# File paths — adjust ROOT_PATH for your environment
ROOT_PATH = './'
DATA_DIR = os.path.join(ROOT_PATH, 'data')

# Google Sheet config
SHEET_ID = '1CLnGu9Gva4Tt2tNHpbwnuxbH2PhnOzvBsYAxn8G0yek'
SHEET_NAME = 'Field_method_to_track_instability_trends_data'

CSV_FILE = os.path.join(ROOT_PATH, 'Field_method_to_track_instability_trends_data.csv')
PRO_FILE = os.path.join(DATA_DIR, 'berthoud_res.pro')
SMET_FILE = os.path.join(DATA_DIR, 'berthoud_res.smet')
AVALANCHE_FILE = os.path.join(DATA_DIR, 'Avalanche_activity.csv')

# AAI weighting (D-Size -> weight)
AAI_WEIGHTS = {
    'D1': 0.1, 'D1.5': 0.32, 'D2': 1.0,
    'D2.5': 3.2, 'D3': 10.0, 'D4': 100.0,
}

# Distance-to-alpha mapping for AAI markers
MAX_DIST_M = 4000
MIN_ALPHA = 0.2

# Plotting
PLOT_DIR = os.path.join(ROOT_PATH, 'plots')

# Time offsets for visual separation: (test_type, removed_WL_cm) -> hours
OFFSET_HOURS = {
    ('PST200', 0): 6,
    ('PST200', 5): 12,
    ('PST100', 0): 18,
}

# Marker shapes per (test_type, removed_WL_cm)
SHAPE_MAP = {
    ('PST200', 0): 'circle',
    ('PST200', 5): 'square',
    ('PST100', 0): 'diamond',
}

# Column lengths per test type
COLUMN_LENGTH = {'PST200': 200, 'PST100': 100}

In [14]:
# --- DATA I/O ---

def parse_pro(filepath):
    """Parse SNOWPACK .pro file into list of dicts with date, heights, densities."""
    data = []
    with open(filepath, 'r') as f:
        in_data = False
        curr = {}
        for line in f:
            if '[DATA]' in line:
                in_data = True
                continue
            if not in_data:
                continue
            parts = line.strip().split(',')
            code = parts[0]
            if code == '0500':
                if curr and 'heights' in curr:
                    data.append(curr)
                try:
                    if parts[1].strip() == 'Date':
                        continue
                    curr = {'date': datetime.strptime(parts[1], '%d.%m.%Y %H:%M:%S')}
                except (ValueError, IndexError):
                    curr = {}
            elif curr and code in ('0501', '0502'):
                vals = np.array([float(x) for x in parts[2:] if x.strip()])
                if code == '0501':
                    curr['heights'] = vals
                else:
                    curr['densities'] = vals
        if curr and 'heights' in curr:
            data.append(curr)
    return data


def load_pst_from_sheet(sheet_id, sheet_name):
    """Load PST data from a Google Sheet (Colab only)."""
    auth.authenticate_user()
    creds, _ = default()
    gc = gspread.authorize(creds)
    sh = gc.open_by_key(sheet_id)
    ws = sh.worksheet(sheet_name)
    rows = ws.get_all_values()
    df = pd.DataFrame(rows[1:], columns=rows[0])
    for col in ['date', 'removed WL (cm)', 'saw cut (cm)', 'crack arrest (cm)',
                'saw distance from end (cm)', 'WL depth', 'HS',
                'load above WL (kg/m^2)', 'period_id']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    return df


def save_pst_to_sheet(df, sheet_id, sheet_name):
    """Write updated HS and load columns back to Google Sheet."""
    creds, _ = default()
    gc = gspread.authorize(creds)
    sh = gc.open_by_key(sheet_id)
    ws = sh.worksheet(sheet_name)

    headers = ws.row_values(1)
    hs_col_idx = headers.index('HS') + 1
    load_col_idx = headers.index('load above WL (kg/m^2)') + 1

    cells_to_update = []
    for i, row in df.iterrows():
        sheet_row = i + 2
        hs_val = '' if pd.isna(row['HS']) else str(row['HS'])
        load_val = '' if pd.isna(row['load above WL (kg/m^2)']) else str(row['load above WL (kg/m^2)'])
        cells_to_update.append(gspread.Cell(sheet_row, hs_col_idx, hs_val))
        cells_to_update.append(gspread.Cell(sheet_row, load_col_idx, load_val))

    ws.update_cells(cells_to_update)
    print(f"Updated {len(cells_to_update)} cells in Google Sheet")


def load_pst_raw(csv_path=None, sheet_id=None, sheet_name=None):
    """Load raw PST dataframe from Google Sheet or CSV."""
    if sheet_id and IN_COLAB:
        print(f"Loading from Google Sheet: {sheet_name}")
        return load_pst_from_sheet(sheet_id, sheet_name)
    else:
        if sheet_id and not IN_COLAB:
            print("Not in Colab \u2014 falling back to CSV")
        return pd.read_csv(csv_path)


def split_into_periods(df):
    """Split PST dataframe into observation periods using the period_id column."""
    valid = df.dropna(subset=['date', 'saw cut (cm)', 'period_id']).copy()
    if valid.empty:
        return []

    valid['date_dt'] = pd.to_datetime(valid['date'].astype(int).astype(str), format='%Y%m%d')
    valid['date_dt'] = valid.apply(
        lambda row: row['date_dt'] + timedelta(
            hours=OFFSET_HOURS.get((row['test'], row['removed WL (cm)']), 0)
        ),
        axis=1,
    )

    periods = []
    for pid in sorted(valid['period_id'].unique()):
        p = valid[valid['period_id'] == pid].copy().reset_index(drop=True)
        if not p.empty:
            periods.append(p)

    return periods


def load_avalanche_activity(filepath):
    """Load avalanche obs, compute daily AAI and distance-based opacity."""
    df = pd.read_csv(filepath)
    df['Date'] = pd.to_datetime(df['Date'])
    df['Weight'] = df['D-Size'].map(AAI_WEIGHTS).fillna(0.1)

    daily = df.groupby('Date').agg(
        AAI=('Weight', 'sum'),
        Mean_Dist=('Distance_to_Plot (m)', 'mean'),
    ).reset_index()

    daily['Alpha'] = 1.0 - (np.clip(daily['Mean_Dist'], 0, MAX_DIST_M) / MAX_DIST_M) * (1.0 - MIN_ALPHA)
    return daily

In [15]:
# --- PROCESSING ---

def compute_load_above_wl(heights, densities, wl_height_cm):
    """Compute SWE load (kg/m\u00b2) above a given weak layer height."""
    load = 0.0
    prev_h = 0.0
    for h, rho in zip(heights, densities):
        if h > wl_height_cm:
            thickness_m = (h - max(prev_h, wl_height_cm)) / 100.0
            load += thickness_m * rho
        prev_h = h
    return load


def compute_snowpack_timeseries(pro_data, wl_height_cm, start_dt, end_dt):
    """Extract HS-above-WL and load timeseries from .pro data for a date window."""
    rows = []
    for d in pro_data:
        if not (start_dt <= d['date'] <= end_dt):
            continue
        if 'densities' not in d:
            continue
        h = d['heights']
        hs_above = h[-1] - wl_height_cm
        load = compute_load_above_wl(h, d['densities'], wl_height_cm)
        rows.append({'date': d['date'], 'hs_above': hs_above, 'load': load})
    return pd.DataFrame(rows)


def get_sf_color(propagation_dist):
    """Map self-fracture propagation distance to a green-ish color."""
    norm = np.clip(propagation_dist / 100.0, 0, 1)
    rgba = plt.cm.RdYlGn(0.5 + 0.5 * norm)
    return f'rgb({int(rgba[0]*255)}, {int(rgba[1]*255)}, {int(rgba[2]*255)})'


def find_nearest_pro(pro_data, pro_dates, target_dt):
    """Find the .pro timestep nearest to target_dt."""
    idx = np.argmin([abs((d - target_dt).total_seconds()) for d in pro_dates])
    return pro_data[idx]


def update_hs_and_load(df, pro_data, pro_dates):
    """Compute HS and load above WL from .pro data for each row.

    Groups by period_id. Within each period, uses the first observation's
    WL depth to establish the WL ground height, then computes HS and load
    for each row from the nearest .pro timestep.
    """
    valid = df.dropna(subset=['date', 'WL depth', 'period_id'])
    if valid.empty:
        return df

    for pid in valid['period_id'].unique():
        mask = (df['period_id'] == pid) & df['date'].notna() & df['WL depth'].notna()
        period_rows = df[mask]
        if period_rows.empty:
            continue

        # WL ground height from first observation in this period
        first = period_rows.iloc[0]
        first_dt = pd.to_datetime(str(int(first['date'])), format='%Y%m%d')
        nearest = find_nearest_pro(pro_data, pro_dates, first_dt)
        wl_height_ground = nearest['heights'][-1] - first['WL depth']

        for idx in period_rows.index:
            row = df.loc[idx]
            row_dt = pd.to_datetime(str(int(row['date'])), format='%Y%m%d')
            pro_snap = find_nearest_pro(pro_data, pro_dates, row_dt)

            if 'densities' not in pro_snap:
                continue

            df.at[idx, 'HS'] = round(pro_snap['heights'][-1], 1)
            df.at[idx, 'load above WL (kg/m^2)'] = round(
                compute_load_above_wl(pro_snap['heights'], pro_snap['densities'], wl_height_ground), 2
            )

    return df

In [16]:
# --- LOAD DATA ---

pro_data = parse_pro(PRO_FILE)
pro_dates = [d['date'] for d in pro_data]
print(f"PRO: {len(pro_data)} timesteps, {pro_dates[0]} to {pro_dates[-1]}")

# Load PST data from Google Sheet or CSV
df_raw = load_pst_raw(csv_path=CSV_FILE, sheet_id=SHEET_ID, sheet_name=SHEET_NAME)

# Update HS and load from .pro data
df_raw = update_hs_and_load(df_raw, pro_data, pro_dates)

# Save updated CSV locally
df_raw.to_csv(CSV_FILE, index=False)
print(f"Updated CSV saved to {CSV_FILE}")

# Write updated values back to Google Sheet
if SHEET_ID and IN_COLAB:
    save_pst_to_sheet(df_raw, SHEET_ID, SHEET_NAME)

# Split into periods for plotting
periods = split_into_periods(df_raw)
print(f"PST: {len(periods)} observation periods")

daily_aai = load_avalanche_activity(AVALANCHE_FILE)
daily_aai['Date'] = pd.to_datetime(daily_aai['Date']) + timedelta(hours=12)
print(f"Avalanche activity: {len(daily_aai)} days with obs")

PRO: 753 timesteps, 2025-09-22 12:00:00 to 2026-03-29 12:00:00
Loading from Google Sheet: Field_method_to_track_instability_trends_data
Updated CSV saved to ./Field_method_to_track_instability_trends_data.csv
Updated 144 cells in Google Sheet
PST: 4 observation periods
Avalanche activity: 17 days with obs


In [17]:
# --- PLOTTING ---

def plot_pst_figure(fig, p_df, daily_aai, plot_start, plot_end):
    """Add PST observations and AAI stars to a plotly figure."""

    # AAI stars
    p_aai = daily_aai[(daily_aai['Date'] >= plot_start) & (daily_aai['Date'] <= plot_end)]
    for _, row in p_aai.iterrows():
        rgba_str = f'rgba(128, 0, 128, {row["Alpha"]:.2f})'
        fig.add_trace(go.Scatter(
            x=[row['Date']], y=[0], mode='markers', showlegend=False,
            marker=dict(symbol='star', size=row['AAI'] * 8 + 15, color=rgba_str,
                        line=dict(color='black', width=1)),
            hovertext=f"AAI: {row['AAI']:.2f}<br>Mean dist: {row['Mean_Dist']:.0f} m",
            hoverinfo='text+x',
        ), secondary_y=False)

    # PST observations
    for _, row in p_df.iterrows():
        dt = row['date_dt']
        sc = row['saw cut (cm)']
        ca = row['crack arrest (cm)']
        atype = str(row['arrest type']).strip()
        sd_end = row['saw distance from end (cm)']
        test_type = row['test']
        col_len = COLUMN_LENGTH.get(test_type, 200)
        marker_symbol = SHAPE_MAP.get((test_type, row['removed WL (cm)']), 'circle')
        lw = 4 if test_type == 'PST100' else 6

        if atype == 'SF':
            color = get_sf_color(ca - sc)
            y_top = ca
            opacity = 1.0
        elif atype == 'End':
            color = 'red'
            y_top = col_len
            opacity = 1.0
        elif atype == 'Arr':
            color = 'orange'
            y_top = min(ca, sc + 100) if not pd.isna(ca) else sc + 100
            opacity = 0.4
        else:
            color = 'gray'
            y_top = sc
            opacity = 0.5

        fig.add_trace(go.Scatter(
            x=[dt, dt], y=[sc, y_top], mode='lines',
            line=dict(color=color, width=lw), opacity=opacity, showlegend=False,
        ), secondary_y=True)

        if atype in ('SF', 'Arr') and not pd.isna(sd_end):
            fig.add_trace(go.Scatter(
                x=[dt, dt], y=[sd_end, col_len], mode='lines',
                line=dict(color=color, width=lw), opacity=opacity, showlegend=False,
            ), secondary_y=True)

        fig.add_trace(go.Scatter(
            x=[dt], y=[sc], mode='markers', showlegend=False,
            marker=dict(symbol=marker_symbol, size=14 if test_type == 'PST200' else 12,
                        color=color, line=dict(color='black', width=1.5)),
        ), secondary_y=True)


def add_legend_traces(fig):
    """Add legend-only dummy traces for arrest types and test markers."""
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='markers',
        marker=dict(symbol='star', color='purple', size=15),
        name='Daily AAI (opacity ~ distance)',
    ), secondary_y=False)
    for label, color, opac in [('End', 'red', 1.0), ('Arr', 'orange', 0.4), ('SF', 'green', 1.0)]:
        fig.add_trace(go.Scatter(
            x=[None], y=[None], mode='lines',
            line=dict(color=color, width=6), opacity=opac,
            name=f'Arrest: {label}',
        ), secondary_y=True)
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='markers',
        marker=dict(symbol='circle', color='gray', size=10, line=dict(color='black', width=1)),
        name='PST200 (0 cm removed)',
    ), secondary_y=True)
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='markers',
        marker=dict(symbol='square', color='gray', size=10, line=dict(color='black', width=1)),
        name='PST200 (5 cm removed)',
    ), secondary_y=True)
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='markers',
        marker=dict(symbol='diamond', color='gray', size=10, line=dict(color='black', width=1)),
        name='PST100 (standard)',
    ), secondary_y=True)

In [ ]:
# --- FULL-SEASON OVERVIEW PLOT ---

os.makedirs(PLOT_DIR, exist_ok=True)

# Combine all periods into one dataframe
all_pst = pd.concat(periods, ignore_index=True)

# Use the first period's WL ground height for the season timeseries
# (this is approximate — WL position is consistent across periods for the same layer)
p200_first = all_pst[all_pst['test'] == 'PST200'].iloc[0]
idx_init = np.argmin([abs((d - p200_first['date_dt']).total_seconds()) for d in pro_dates])
wl_height_season = pro_data[idx_init]['heights'][-1] - p200_first['WL depth']

# Full-season snowpack timeseries — from first snow to end of .pro
first_snow_idx = next(i for i, d in enumerate(pro_data) if 'densities' in d and d['heights'][-1] > 5)
season_start = pro_data[first_snow_idx]['date']
season_end = pro_dates[-1]
res_df = compute_snowpack_timeseries(pro_data, wl_height_season, season_start, season_end)

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(go.Scatter(
    x=res_df['date'], y=res_df['hs_above'],
    name='HS above WL (cm)', line=dict(color='blue', width=3),
), secondary_y=False)
fig.add_trace(go.Scatter(
    x=res_df['date'], y=res_df['load'],
    name='Load above WL (kg/m\u00b2)', line=dict(color='darkblue', width=2, dash='dash'),
), secondary_y=False)

add_legend_traces(fig)
plot_pst_figure(fig, all_pst, daily_aai, season_start, season_end)

fig.update_layout(
    title=f'Full Season Overview (WL height from ground: {wl_height_season:.1f} cm)',
    template='plotly_white', height=800,
)
fig.update_yaxes(title_text="HS (cm) / Load (kg/m\u00b2) above WL", secondary_y=False)
fig.update_yaxes(title_text="PST Distance (cm)", range=[-10, 210], secondary_y=True)

fig.write_html(os.path.join(PLOT_DIR, 'stability_trend_full_season.html'))
fig.show()
print(f"Full season: plotted ({len(all_pst)} total obs)")

In [18]:
# --- FULL-SEASON OVERVIEW PLOT ---

os.makedirs(PLOT_DIR, exist_ok=True)

# Combine all periods into one dataframe
all_pst = pd.concat(periods, ignore_index=True)

# Use the first period's WL ground height for the season timeseries
# (this is approximate — WL position is consistent across periods for the same layer)
p200_first = all_pst[all_pst['test'] == 'PST200'].iloc[0]
idx_init = np.argmin([abs((d - p200_first['date_dt']).total_seconds()) for d in pro_dates])
wl_height_season = pro_data[idx_init]['heights'][-1] - p200_first['WL depth']

# Full-season snowpack timeseries — from first snow to end of .pro
first_snow_idx = next(i for i, d in enumerate(pro_data) if 'densities' in d and d['heights'][-1] > 5)
season_start = pro_data[first_snow_idx]['date']
season_end = pro_dates[-1]
res_df = compute_snowpack_timeseries(pro_data, wl_height_season, season_start, season_end)

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(go.Scatter(
    x=res_df['date'], y=res_df['hs_above'],
    name='HS above WL (cm)', line=dict(color='blue', width=3),
), secondary_y=False)
fig.add_trace(go.Scatter(
    x=res_df['date'], y=res_df['load'],
    name='Load above WL (kg/m\u00b2)', line=dict(color='darkblue', width=2, dash='dash'),
), secondary_y=False)

add_legend_traces(fig)
plot_pst_figure(fig, all_pst, daily_aai, season_start, season_end)

fig.update_layout(
    title=f'Full Season Overview (WL height from ground: {wl_height_season:.1f} cm)',
    template='plotly_white', height=800,
)
fig.update_yaxes(title_text="HS (cm) / Load (kg/m\u00b2) above WL", secondary_y=False)
fig.update_yaxes(title_text="PST Distance (cm)", range=[-10, 210], secondary_y=True)

fig.write_html(os.path.join(PLOT_DIR, 'stability_trend_full_season.html'))
fig.show()
print(f"Full season: plotted ({len(all_pst)} total obs)")

Full season: plotted (69 total obs)


In [19]:
# --- STATISTICS ---

def compute_stats(periods, daily_aai):
    """Compute summary statistics for PST observations."""
    all_pst = pd.concat(periods, ignore_index=True)
    avy_dates = set(daily_aai['Date'].dt.date)

    all_pst['date_cal'] = all_pst['date_dt'].dt.date
    all_pst['avy_day'] = all_pst['date_cal'].isin(avy_dates)
    all_pst['prop_dist'] = all_pst['crack arrest (cm)'] - all_pst['saw cut (cm)']

    def test_label(row):
        if row['test'] == 'PST100':
            return 'PST100'
        return f"PST200 ({int(row['removed WL (cm)'])}cm)"
    all_pst['test_cat'] = all_pst.apply(test_label, axis=1)

    report = []
    report.append("# PST Statistics Report")
    report.append(f"\nTotal observations: {len(all_pst)} across {len(periods)} periods")
    avy_day_count = all_pst.drop_duplicates('date_cal')['avy_day'].sum()
    nonavy_day_count = (~all_pst.drop_duplicates('date_cal')['avy_day']).sum()
    report.append(f"Observation days with nearby avalanches: {avy_day_count}")
    report.append(f"Observation days without nearby avalanches: {nonavy_day_count}")

    # --- Basic stats tables ---
    for metric_name, col in [('Saw Cut Length (cm)', 'saw cut (cm)'),
                              ('Propagation Distance (cm)', 'prop_dist'),
                              ('Saw Distance from End (cm)', 'saw distance from end (cm)')]:
        report.append(f"\n## {metric_name}")
        report.append(f"\n{'Test':<16} {'Context':<14} {'n':>3} {'Mean':>7} {'Std':>7} {'Min':>5} {'Max':>5}")
        report.append("-" * 62)
        for tc in ['PST200 (0cm)', 'PST200 (5cm)', 'PST100']:
            for avy_label, avy_val in [('Avy days', True), ('Non-avy days', False)]:
                s = all_pst[(all_pst['test_cat'] == tc) & (all_pst['avy_day'] == avy_val)][col].dropna()
                if len(s) > 0:
                    report.append(f"{tc:<16} {avy_label:<14} {len(s):>3} {s.mean():>7.1f} {s.std():>7.1f} {s.min():>5.0f} {s.max():>5.0f}")
                else:
                    report.append(f"{tc:<16} {avy_label:<14}   0")

    # --- Arrest type counts ---
    report.append("\n## Arrest Type Counts")
    report.append(f"\n{'Test':<16} {'Context':<14} {'n':>3}  {'End':>4} {'Arr':>4} {'SF':>4}")
    report.append("-" * 52)
    for tc in ['PST200 (0cm)', 'PST200 (5cm)', 'PST100']:
        for avy_label, avy_val in [('Avy days', True), ('Non-avy days', False)]:
            s = all_pst[(all_pst['test_cat'] == tc) & (all_pst['avy_day'] == avy_val)]
            counts = s['arrest type'].value_counts()
            report.append(f"{tc:<16} {avy_label:<14} {len(s):>3}  {counts.get('End',0):>4} {counts.get('Arr',0):>4} {counts.get('SF',0):>4}")

    # --- Paired state classification ---
    report.append("\n## Paired State Classification (PST200: 0cm vs 5cm)")
    pair_states = []
    for date_cal in sorted(all_pst['date_cal'].unique()):
        grp = all_pst[(all_pst['date_cal'] == date_cal) & (all_pst['test'] == 'PST200')]
        r0 = grp[grp['removed WL (cm)'] == 0]
        r5 = grp[grp['removed WL (cm)'] == 5]
        if r0.empty or r5.empty:
            continue
        a0 = str(r0.iloc[0]['arrest type']).strip() if pd.notna(r0.iloc[0]['arrest type']) else '?'
        a5 = str(r5.iloc[0]['arrest type']).strip() if pd.notna(r5.iloc[0]['arrest type']) else '?'
        load = r0.iloc[0].get('load above WL (kg/m^2)', np.nan)
        is_avy = date_cal in avy_dates

        if a0 == 'End' and a5 == 'End':
            state = 'UNSTABLE (both End)'
        elif a0 in ('SF', 'Arr') and a5 == 'End':
            state = 'NEAR THRESHOLD'
        elif a0 == 'SF' and a5 == 'SF':
            state = 'STABLE (both SF)'
        elif a0 == 'Arr' and a5 == 'Arr':
            state = 'MARGINAL (both Arr)'
        else:
            state = f'Other ({a0}/{a5})'

        pair_states.append({'date': date_cal, 'state': state, 'load': load, 'avy_day': is_avy})

    ps_df = pd.DataFrame(pair_states)

    for avy_label, avy_val in [('Avalanche days', True), ('Non-avalanche days', False)]:
        sub = ps_df[ps_df['avy_day'] == avy_val]
        report.append(f"\n{avy_label} ({len(sub)} days):")
        counts = sub['state'].value_counts()
        for s in ['UNSTABLE (both End)', 'NEAR THRESHOLD', 'STABLE (both SF)', 'MARGINAL (both Arr)']:
            c = counts.get(s, 0)
            pct = c / len(sub) * 100 if len(sub) > 0 else 0
            report.append(f"  {s:<30s}  {c:>2d}  ({pct:.0f}%)")
        # Any others
        for s, c in counts.items():
            if s not in ['UNSTABLE (both End)', 'NEAR THRESHOLD', 'STABLE (both SF)', 'MARGINAL (both Arr)']:
                pct = c / len(sub) * 100
                report.append(f"  {s:<30s}  {c:>2d}  ({pct:.0f}%)")

    # --- PST100 vs PST200 agreement ---
    report.append("\n## PST100 vs PST200 (0cm) Agreement")
    report.append("\nThis quantifies how often the standard 100cm PST saturates at End")
    report.append("when the 200cm PST shows more nuanced results.")
    agreements = []
    for date_cal in sorted(all_pst['date_cal'].unique()):
        grp = all_pst[all_pst['date_cal'] == date_cal]
        r200 = grp[(grp['test'] == 'PST200') & (grp['removed WL (cm)'] == 0)]
        r100 = grp[grp['test'] == 'PST100']
        if r200.empty or r100.empty:
            continue
        a200 = str(r200.iloc[0]['arrest type']).strip()
        a100 = str(r100.iloc[0]['arrest type']).strip()
        agreements.append({'PST200_0cm': a200, 'PST100': a100})

    ag_df = pd.DataFrame(agreements)
    for a200_type in ['End', 'Arr', 'SF']:
        sub = ag_df[ag_df['PST200_0cm'] == a200_type]
        if sub.empty:
            continue
        end_count = (sub['PST100'] == 'End').sum()
        report.append(f"\n  PST200(0cm) = {a200_type}:")
        report.append(f"    PST100 End: {end_count}/{len(sub)} ({end_count/len(sub)*100:.0f}%)")
        for a100_type in ['End', 'Arr', 'SF']:
            c = (sub['PST100'] == a100_type).sum()
            if c > 0 and a100_type != 'End':
                report.append(f"    PST100 {a100_type}: {c}/{len(sub)} ({c/len(sub)*100:.0f}%)")

    # --- 5cm removal sensitivity ---
    report.append("\n## Sensitivity of 5cm WL Removal")
    sc_diffs = []
    for date_cal in sorted(all_pst['date_cal'].unique()):
        grp = all_pst[(all_pst['date_cal'] == date_cal) & (all_pst['test'] == 'PST200')]
        r0 = grp[grp['removed WL (cm)'] == 0]
        r5 = grp[grp['removed WL (cm)'] == 5]
        if r0.empty or r5.empty:
            continue
        sc_diffs.append(r5.iloc[0]['saw cut (cm)'] - r0.iloc[0]['saw cut (cm)'])

    sc_arr = np.array(sc_diffs)
    report.append(f"\nSaw cut difference (5cm removal - standard):")
    report.append(f"  mean={sc_arr.mean():.1f}  std={sc_arr.std():.1f}  range={sc_arr.min():.0f} to {sc_arr.max():.0f} cm")
    report.append(f"  (negative = 5cm removal needed shorter cut, as expected)")
    report.append(f"  Consistent direction (5cm <= 0cm): {(sc_arr <= 0).sum()}/{len(sc_arr)} ({(sc_arr <= 0).sum()/len(sc_arr)*100:.0f}%)")

    return "\n".join(report), ps_df


report_text, paired_states = compute_stats(periods, daily_aai)
print(report_text)

# Save report
report_path = os.path.join(PLOT_DIR, 'pst_statistics_report.md')
with open(report_path, 'w') as f:
    f.write(report_text)
print(f"\nReport saved to {report_path}")

# PST Statistics Report

Total observations: 69 across 4 periods
Observation days with nearby avalanches: 8
Observation days without nearby avalanches: 15

## Saw Cut Length (cm)

Test             Context          n    Mean     Std   Min   Max
--------------------------------------------------------------
PST200 (0cm)     Avy days         8    58.8    35.1    33   142
PST200 (0cm)     Non-avy days    15    51.7    40.5    25   162
PST200 (5cm)     Avy days         8    46.6    21.9    27    98
PST200 (5cm)     Non-avy days    15    65.5    41.2    22   158
PST100           Avy days         8    48.0    10.4    34    65
PST100           Non-avy days    15    43.0    15.8    22    73

## Propagation Distance (cm)

Test             Context          n    Mean     Std   Min   Max
--------------------------------------------------------------
PST200 (0cm)     Avy days         5    66.0    54.1    22   159
PST200 (0cm)     Non-avy days    13    26.4    45.9     0   163
PST200 (5cm)     Avy da

In [20]:
# --- PER-PERIOD PLOTS ---

for i, p_df in enumerate(periods):
    p200 = p_df[p_df['test'] == 'PST200']
    if p200.empty:
        continue

    if p200['date_dt'].min() < min(pro_dates) or p200['date_dt'].max() > max(pro_dates):
        print(f"Period {i+1}: skipped (outside model range)")
        continue

    first_obs = p200.iloc[0]
    idx_init = np.argmin([abs((d - first_obs['date_dt']).total_seconds()) for d in pro_dates])
    wl_height_ground = pro_data[idx_init]['heights'][-1] - first_obs['WL depth']

    # Use PST200 dates for the plot window (PST100 dates may have typos)
    plot_start = p200['date_dt'].min() - timedelta(days=2)
    plot_end = p200['date_dt'].max() + timedelta(days=2)

    res_df = compute_snowpack_timeseries(pro_data, wl_height_ground, plot_start, plot_end)
    if res_df.empty:
        print(f"Period {i+1}: no model data in window")
        continue

    fig = make_subplots(specs=[[{"secondary_y": True}]])

    fig.add_trace(go.Scatter(
        x=res_df['date'], y=res_df['hs_above'],
        name='HS above WL (cm)', line=dict(color='blue', width=3),
    ), secondary_y=False)
    fig.add_trace(go.Scatter(
        x=res_df['date'], y=res_df['load'],
        name='Load above WL (kg/m\u00b2)', line=dict(color='darkblue', width=2, dash='dash'),
    ), secondary_y=False)

    add_legend_traces(fig)
    plot_pst_figure(fig, p_df, daily_aai, plot_start, plot_end)

    fig.update_layout(
        title=f'Stability Trend Period {i+1} (WL height from ground: {wl_height_ground:.1f} cm)',
        template='plotly_white', height=800,
    )
    fig.update_yaxes(title_text="HS (cm) / Load (kg/m\u00b2) above WL", secondary_y=False)
    fig.update_yaxes(title_text="PST Distance (cm)", range=[-10, 210], secondary_y=True)

    fig.write_html(os.path.join(PLOT_DIR, f'stability_trend_period_{i+1}.html'))
    fig.show()
    print(f"Period {i+1}: plotted ({len(p_df)} obs, WL @ {wl_height_ground:.1f} cm)")

Period 1: plotted (15 obs, WL @ 34.8 cm)


Period 2: plotted (15 obs, WL @ 36.7 cm)


Period 3: plotted (24 obs, WL @ 35.1 cm)


Period 4: plotted (15 obs, WL @ 34.0 cm)
